# self-rag

# Resources

https://github.com/sunnysavita10/langgraph-end-to-end/blob/main/agent_based_rag/self_rag.ipynb

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import EnsembleRetriever, BM25Retriever
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain.vectorstores import Chroma
import pdfplumber
import os

In [ ]:
from dotenv import load_dotenv
load_dotenv()

api_key = os.environ['UNIFIED_LLM_KEY']
# print(api_key)
base_url = ""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=256,
    api_key=api_key,
    base_url=base_url
)

In [ ]:
llm.invoke("Hi, How are you?")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs

In [ ]:
from ml_server_embedding import get_embeddings

embeddings = get_embeddings()

docs_list = [item for sublist in docs for item in sublist]
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100, chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs_list)

# Add to vectorDB
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding=embeddings,
)

retriever = vectorstore.as_retriever()

In [ ]:
type(retriever)

In [ ]:
docs = retriever.get_relevant_documents(question)

In [ ]:
from langchain.tools.retriever import create_retriever_tool

retrieval_tool = create_retriever_tool(
    retriever,
    "retrieve_blog_posts",
    "Search and return information about Lilian Weng blog posts on LLM agents, prompt engineering, and adversarial attacks on LLMs.",
)

tools = [retrieval_tool]

In [ ]:
type(retrieval_tool)

## Let's look into the retriever grader

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

def GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""
    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

In [ ]:
structured_llm_grader = llm.with_structured_output(GradeDocuments)

In [ ]:
# Prompt
system = """You are a grader checking if a document is relevant to a user's question.The check has to be done very strictly..  
If the document has words or meanings related to the question, mark it as relevant.  
Give a simple 'yes' or 'no' answer to show if the document is relevant or not."""

grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

In [ ]:
my_retrieval_grader = grade_prompt | structured_llm_grader
question = "what is ai agent?"
docs = retriever.get_relevant_documents(question)
docs

In [ ]:
doc_txt = docs[2].page_content

In [ ]:
print(my_retrieval_grader.invoke({"document":doc_txt,"question": question}))

## let's look into the data generation

In [ ]:
from langchain_core.output_parsers import StrOutputParser

generation_prompt=ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""
)

In [ ]:
from langchain_core.runnables import RunnableLambda
rag_chain = generation_prompt | llm

In [ ]:
generation = rag_chain.invoke({"context": docs, "question": question})
generation

## Hallucination Grader

In [ ]:
# Data model
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score: str = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )

In [ ]:
structured_llm_grader = llm.with_structured_output(GradeHallucinations)

In [ ]:
# Prompt
system = """You are a grader checking if an LLM generation is grounded in or supported by a set of retrieved facts.  
Give a simple 'yes' or 'no' answer. 'Yes' means the generation is grounded in or supported by a set of retrieved the facts."""
hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}"),
    ]
)

In [ ]:
hallucinations_grader = hallucination_prompt | structured_llm_grader
print(hallucinations_grader.invoke({"documents": docs, "generation": generation}))

## Answer Grader

In [ ]:
### Answer Grader
# Data model
class GradeAnswer(BaseModel):
    """Binary score to assess answer addresses question."""

    binary_score: str = Field(
        description="Answer addresses the question, 'yes' or 'no'"
    )


# LLM with function call
structured_llm_grader = llm.with_structured_output(GradeAnswer)

# Prompt
system = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question."""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structured_llm_grader
print(answer_grader.invoke({"question": question, "generation": generation}))

## Question Re-writer

In [ ]:
system = """You are a question re-writer that converts an input question into a better optimized version for vector store retrieval document.  
You are given both a question and a document.  
- First, check if the question is relevant to the document by identifying a connection or relevance between them.  
- If there is a little relevancy, rewrite the question based on the semantic intent of the question and the context of the document.  
- If no relevance is found, simply return this single word "question not relevant." dont return the entire phrase 
Your goal is to ensure the rewritten question aligns well with the document for better retrieval."""

re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human","""Here is the initial question: \n\n {question} \n,
             Here is the document: \n\n {documents} \n ,
             Formulate an improved question. if possible other return 'question not relevant'."""
        ),
    ]
)
question_rewriter = re_write_prompt | llm | StrOutputParser()

In [ ]:
question="who is a current indian prime minister?"
question_rewriter.invoke({"question":question,"documents":docs})

In [ ]:
from typing import List
from typing_extensions import TypedDict

class AgentState(TypedDict):
    question: str
    generation: str
    documents: List[str]
    filtered_document: List[str]
    unfiltered_document: List[str]
    quality_check: str  # Added to store quality check result

In [ ]:
def retrieve(state:AgentState):
    print("----RETRIEVE----")
    question=state['question']
    documents=retriever.get_relevant_documents(question)
    return {"documents": documents, "question": question}

In [ ]:
def grade_documents(state:AgentState):
    print("----CHECK DOCUMENTS RELEVANCE TO THE QUESTION----")
    question = state['question']
    documents = state['documents']

    filtered_docs = []
    unfiltered_docs = []
    for doc in documents:
        score=my_retrieval_grader.invoke({"question":question, "document":doc})
        grade=score['binary_score']  # FIXED: Changed from score.binary_score to dict access

        if grade=='yes':
            print("----GRADE: DOCUMENT RELEVANT----")
            filtered_docs.append(doc)
        else:
            print("----GRADE: DOCUMENT NOT RELEVANT----")
            unfiltered_docs.append(doc)
    if len(unfiltered_docs)>1:
        return {"unfilter_documents": unfiltered_docs,"filter_documents":[], "question": question}
    else:
        return {"filter_documents": filtered_docs,"unfilter_documents":[],"question": question}

In [ ]:
def decide_to_generate(state:AgentState):
    print("----ACCESS GRADED DOCUMENTS----")
    state["question"]
    unfiltered_documents = state["unfilter_documents"]
    filtered_documents = state["filter_documents"]


    if unfiltered_documents:
        print("----ALL THE DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY----")
        return "transform_query"
    if filtered_documents:
        print("----DECISION: GENERATE----")
        return "generate"

In [ ]:
def generate(state:AgentState):
    print("----GENERATE----")
    question=state["question"]
    documents=state["documents"]

    generation = rag_chain.invoke({"context": documents,"question":question})
    return {"documents":documents,"question":question,"generation":generation}

In [ ]:
from langgraph.graph import END, StateGraph, START
def transform_query(state:AgentState):
    question=state["question"]
    documents=state["documents"]

    print(f"this is my document{documents}")
    response = question_rewriter.invoke({"question":question,"documents":documents})
    print(f"----RESPONSE---- {response}")
    if response == 'question not relevant':
        print("----QUESTION IS NOT AT ALL RELEVANT----")
        return {"documents":documents,"question":response,"generation":"question was not at all relevant"}
    else:   
        return {"documents":documents,"question":response}

In [ ]:
def decide_to_generate_after_transformation(state:AgentState):
    question=state["question"]

    if question=="question not relevant":
        return "query_not_at_all_relevant"
    else:
        return "Retriever"

In [ ]:
import pprint

# Node function - performs the check and returns dict
def check_generation_quality(state:AgentState):
    print("---CHECK HALLUCINATIONS---")
    question= state['question']
    documents = state['documents']
    generation = state["generation"]

    score = hallucinations_grader.invoke({"documents":documents,"generation":generation})
    grade = score['binary_score']  # FIXED: Changed from score.binary_score to dict access

    # Store the quality check result in state
    quality_result = "not useful"

    # Check hallucinations
    if grade=='yes':
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")
        print("---GRADE GENERATION vs QUESTION ---")

        score = answer_grader.invoke({"question":question,"generation":generation})
        grade = score['binary_score']  # FIXED: Changed from score.binary_score to dict access

        if grade=='yes':
            print("---DECISION: GENERATION ADDRESS THE QUESTION ---")
            quality_result = "useful"
        else:
            print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION, RE-TRY---TRANSFORM QUERY")
            quality_result = "not useful"
    else:
        print("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---TRANSFORM QUERY")
        quality_result = "not useful"

    return {"quality_check": quality_result}

# Routing function - decides next step based on quality
def route_based_on_quality(state:AgentState) -> str:
    quality_result = state.get("quality_check", "not useful")
    return quality_result

## Build LangGraph

In [ ]:
from langgraph.graph import START, END

workflow = StateGraph(AgentState)

# Add nodes - these are the ACTUAL node names
workflow.add_node("Vector Retrieval", retrieve)
workflow.add_node("Grade Retrieved Document", grade_documents)
workflow.add_node("Content Generator", generate)
workflow.add_node("Transform User Query", transform_query)
workflow.add_node("Check Generation Quality", check_generation_quality)  # FIXED: Using node function

# Add edges - Start -> Retrieve -> Grade
workflow.add_edge(START, "Vector Retrieval")
workflow.add_edge("Vector Retrieval", "Grade Retrieved Document")

# Conditional edge from Grade Retrieved Document
workflow.add_conditional_edges(
    "Grade Retrieved Document",
    decide_to_generate,
    {
        "transform_query": "Transform User Query",
        "generate": "Content Generator"
    }
)

# Add edge from Content Generator to Check Generation Quality
workflow.add_edge("Content Generator", "Check Generation Quality")

# Conditional edge to check generation quality - FIXED: Using routing function
workflow.add_conditional_edges(
    "Check Generation Quality",
    route_based_on_quality,  # FIXED: Use routing function that returns string
    {
        "useful": END,
        "not useful": "Transform User Query"
    }
)

# Conditional edge from Transform User Query
workflow.add_conditional_edges(
    "Transform User Query",
    decide_to_generate_after_transformation,
    {
        "Retriever": "Vector Retrieval",
        "query_not_at_all_relevant": END
    }
)

graph = workflow.compile()